### 1. Background & Motivation 

This notebook investigates pipeline failure behavior

#### Goal of This Notebook
Not all iris images can be successfully processed by the pipeline. Images may fail due to poor quality, occlusion, segmentation errors, or other issues. The goal is to:
- Run the IRIS pipeline over two standard iris datasets (CASIA Iris Thousand and IITD)
- Track which images succeed and which fail, along with the error message
- Build a labeled dataset (`success=1` / `success=0`) that can be used for failure analysis

#### Datasets

**CASIA-Iris-Thousand** : 20,000 images from 1,000 subjects captured in near-infrared (NIR) by the Chinese Academy of Sciences 

**IITD**: Iris images collected at IIT Delhi, widely used for benchmarking iris recognition systems

### 2. Imports
Core dependencies:
- **`iris`** — Worldcoin's IRIS pipeline package for iris segmentation and encoding
- **`cv2`** — OpenCV for reading images as grayscale arrays (required format for the pipeline)
- **`pandas` / `numpy`** — Data manipulation and array operations
- **`sklearn`** — Imported here for potential downstream use in training a failure prediction classifier
- **`time`** — Used to benchmark per-image pipeline runtime

In [ ]:
import os
import time 
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import iris

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

### 3. Data Collection
We define path-collection functions for each dataset. Rather than loading images into memory upfront, we first build a lightweight index DataFrame containing:
- `subject_id` — The identity label for each subject
- `eye` — Which eye (`L` / `R` / `unknown`), inferred from folder structure or filename
- `image_path` — Full path to the image file on disk
- `dataset` — Source dataset tag (`CASIA` or `IITD`)

#### Directory Structure Handling
The two datasets have different folder layouts:
- **CASIA** uses a two-level structure: `subject_id/eye_side/image.jpg`, but also supports a flat fallback where eye side is inferred from the filename.
- **IITD** uses a flat structure: `subject_id/image.jpg`, with eye side inferred from the filename.

Both functions skip hidden files (e.g. `.DS_Store`) and only index supported image formats (`.jpg`, `.jpeg`, `.png`, `.bmp`).

In [ ]:
CASIA_ROOT = "./CASIA-Iris-Thousand/CASIA-Iris-Thousand"
IITD_ROOT = "./IITD_database"

In [ ]:
CASIA_ROOT = "./CASIA-Iris-Thousand/CASIA-Iris-Thousand"
IITD_ROOT = "./IITD_database"
def collect_casia_paths(root):
    data = []

    for subject in os.listdir(root):
        if subject.startswith("."):
            continue

        subject_path = os.path.join(root, subject)
        if not os.path.isdir(subject_path):
            continue

        found_nested_dirs = False

        for item in os.listdir(subject_path):
            if item.startswith("."):
                continue
            item_path = os.path.join(subject_path, item)

            if os.path.isdir(item_path):
                found_nested_dirs = True
                for img in os.listdir(item_path):
                    if img.startswith("."):
                        continue
                    if img.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                        data.append({
                            "subject_id": str(subject),
                            "eye": str(item),
                            "image_path": os.path.join(item_path, img),
                            "dataset": "CASIA"})

        if not found_nested_dirs:
            for img in os.listdir(subject_path):
                if img.startswith("."):
                    continue
                if img.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                    filename = img.lower()
                    if "_l" in filename or "left" in filename:
                        eye = "L"
                    elif "_r" in filename or "right" in filename:
                        eye = "R"
                    else:
                        eye = "unknown"

                    data.append({
                        "subject_id": str(subject),
                        "eye": eye,
                        "image_path": os.path.join(subject_path, img),
                        "dataset": "CASIA"})

    return pd.DataFrame(data)

In [ ]:
def collect_iitd_paths(root):
    data = []

    for subject in os.listdir(root):
        if subject.startswith("."):
            continue

        subject_path = os.path.join(root, subject)
        if not os.path.isdir(subject_path):
            continue

        for img in os.listdir(subject_path):
            if img.startswith("."):
                continue
            if img.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                filename = img.lower()
                if "_l" in filename or "left" in filename:
                    eye = "L"
                elif "_r" in filename or "right" in filename:
                    eye = "R"
                else:
                    eye = "unknown"

                data.append({
                    "subject_id": str(subject),
                    "eye": eye,
                    "image_path": os.path.join(subject_path, img),
                    "dataset": "IITD"})

    return pd.DataFrame(data)

In [ ]:
casia_df = collect_casia_paths(CASIA_ROOT)
iitd_df = collect_iitd_paths(IITD_ROOT)

print("CASIA:")
print(casia_df.head())
print(len(casia_df), "images")

print("\nIITD:")
print(iitd_df.head())
print(len(iitd_df), "images")

In [ ]:
print("CASIA subjects:", casia_df["subject_id"].nunique())
print("IITD subjects:", iitd_df["subject_id"].nunique())

print("\nCASIA eye counts:")
print(casia_df["eye"].value_counts(dropna=False))

print("\nIITD eye counts:")
print(iitd_df["eye"].value_counts(dropna=False))

### 4. Iris Pipeline Encoding
Here we run each image through the IRIS pipeline and split the results into **successes** and **failures**.

#### Pipeline Initialization
`iris.IRISPipeline()` loads the full Worldcoin pipeline including the segmentation model, normalization, and IrisCode encoder. This is initialized once and reused across all images to avoid repeated overhead.

#### Template Extraction Helper
`extract_template_from_result()` is a utility that safely retrieves the IrisCode template from the pipeline output, handling variation in the output schema (the template may be keyed as `iris_template`, `template`, or `iris_code` depending on the pipeline version).

#### `encode_dataset()` — Core Encoding Loop
This function iterates over every image in the index DataFrame and:
1. Reads the image as a **grayscale array** using OpenCV (required by the pipeline)
2. Constructs an `iris.IRImage` object with the image data, filename, and eye side
3. Passes it through the pipeline and records the **wall-clock runtime**
4. If the pipeline returns no error → appends to `records` (successes) with the IrisCode template
5. If the pipeline returns an error or an exception is raised → appends to `failures` with the error message

The function returns two DataFrames: one for successful encodings and one for failures.

#### Sampling
To keep runtime manageable, we draw a random sample of **2,000 images per dataset** (4,000 total) with a fixed random seed for reproducibility.

In [ ]:
pipeline = iris.IRISPipeline()

In [ ]:
def extract_template_from_result(result):
    if isinstance(result, dict):
        if "iris_template" in result:
            return result["iris_template"]
        if "template" in result:
            return result["template"]
        if "iris_code" in result:
            return result["iris_code"]
    if hasattr(result, "iris_template"):
        return result.iris_template
    if hasattr(result, "template"):
        return result.template
    if hasattr(result, "iris_code"):
        return result.iris_code

    raise ValueError("Could not find template in pipeline result.")

In [ ]:
def encode_dataset(df, pipeline, limit=None):
    records = []
    failures = []

    if limit is not None:
        df = df.head(limit).copy()

    for _, row in df.iterrows():
        try:
            img = cv2.imread(row["image_path"], cv2.IMREAD_GRAYSCALE)
            if img is None:
                raise ValueError("Could not read image")

            eye_side = "left" if row["eye"] == "L" else "right" if row["eye"] == "R" else "left"

            start = time.time()

            output = pipeline(
                iris.IRImage(
                    img_data=img,
                    image_id=os.path.basename(row["image_path"]),
                    eye_side=eye_side))

            runtime = time.time() - start

            if output.get("error") is None:
                records.append({
                    "subject_id": row["subject_id"],
                    "eye": row["eye"],
                    "image_path": row["image_path"],
                    "dataset": row["dataset"],
                    "template": output["iris_template"],
                    "runtime": runtime})
            else:
                failures.append({
                    "subject_id": row["subject_id"],
                    "eye": row["eye"],
                    "image_path": row["image_path"],
                    "dataset": row["dataset"],
                    "error": str(output["error"])})

        except Exception as e:
            failures.append({
                "subject_id": row["subject_id"],
                "eye": row["eye"],
                "image_path": row["image_path"],
                "dataset": row["dataset"],
                "error": str(e)})

    return pd.DataFrame(records), pd.DataFrame(failures)

In [ ]:
casia_sample = casia_df.sample(n=2000, random_state=42)
iitd_sample = iitd_df.sample(n=2000, random_state=42)

casia_encoded, casia_failures = encode_dataset(casia_sample, pipeline)
iitd_encoded, iitd_failures = encode_dataset(iitd_sample, pipeline)

print("CASIA encoded:", len(casia_encoded))
print("CASIA failures:", len(casia_failures))
print("IITD encoded:", len(iitd_encoded))
print("IITD failures:", len(iitd_failures))

### 5. Create Failure Dataset

We now consolidate the encoding results from both datasets into a single labeled DataFrame (failure dataset) which will serve as the foundation for downstream analysis.

Here are the following columns:

| Column | Description |
|---|---|
| `dataset` | Source dataset (`CASIA` or `IITD`) |
| `subject_id` | Subject identity label |
| `eye` | Eye side (`L`, `R`, or `unknown`) |
| `image_path` | Full path to the image file |
| `success` | Binary label: `1` = pipeline succeeded, `0` = pipeline failed |
| `error` | Error message string for failures; `None` for successes |

Duplicates are dropped by `image_path` to ensure each image appears exactly once.

#### Output
The final DataFrame is exported to **`failure_dataset.csv`** for use in future notebooks, such as extracting image-level features and training a failure prediction model.

In [ ]:
def build_failure_dataset(casia_encoded, casia_failures, iitd_encoded, iitd_failures):
    success_frames = []
    failure_frames = []

    for df in [casia_encoded, iitd_encoded]:
        if len(df) > 0:
            temp = df[["dataset", "subject_id", "eye", "image_path"]].copy()
            temp["success"] = 1
            temp["error"] = None
            success_frames.append(temp)

    for df in [casia_failures, iitd_failures]:
        if len(df) > 0:
            temp = df[["dataset", "subject_id", "eye", "image_path", "error"]].copy()
            temp["success"] = 0
            failure_frames.append(temp)

    combined = pd.concat(success_frames + failure_frames, ignore_index=True)
    combined = combined.drop_duplicates(subset=["image_path"]).reset_index(drop=True)

    return combined

In [ ]:
failure_df = build_failure_dataset(casia_encoded, casia_failures,iitd_encoded, iitd_failures)
failure_df["success"].value_counts()

In [ ]:
failure_df.to_csv("failure_dataset.csv", index=False)

print("Saved CSV files:")
print("failure_dataset.csv")